In [ ]:
%pip install tensorflow

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# Quantum Neural Networks

A Quantum Neural Network (QNN) is a quantum machine learning model inspired by classical neural networks. Analogous to their classical counterparts, QNNs are fed with a classical input in the form of an array of numbers. This data then flows through a sequence of layers that perform transformations based on optimizable parameters. Finally, the processed data is output through a final layer. Let's see how QNNs apply these three stages:

**1. Data preparation**: in order to transform the classical inputs in form of arrays into quantum states, we need to design and apply a feature map.

**2. Data processing**: this stage consist of the application of a variational circuit, that is a quantum circuit that depends on some optimizable parameters. These variational circuits perform transformations in the input data (that have already been embedded into quantum states) and are usually structured in layers, as in classical neural networks.

**3. Data output**: in order to obtain a classical output we need to perform a quantum measurement to obtain the expectation values or probabilities.

These three stages allow us to design different architectures for our QNNs, making it essential to be intentional about every choice. The feature map, the ansatz, and the measurement operation must be tailored to our data characteristics and specific goals, as they will exert a critical influence on the model's performance. Data must be normalized according to the specific feature map in use, and the number of parameters must be kept as close as possible to the sweet spot between underfitting and overfitting.

The most typical features maps for QNNs are:

* **Angle encoding**: each classical feature is encoded as the angle of a quantum rotation gate. For example, a data point $x = (x_1, x_2, \dots, x_n)$ can be encoded by applying rotations such as $R_y(x_i)$ or $R_z(x_i)$ to different qubits. This is one of the simplest and most widely used encoding strategies, but the input data usually need to be normalized to a suitable angular range, such as $[0,\pi]$ or $[0,2\pi]$.

* **Amplitude encoding**: the classical data vector is encoded directly into the amplitudes of a quantum state. Given a normalized vector $x$, the corresponding quantum state can be written as

$$
|\psi(x)\rangle = \sum_i x_i |i\rangle.
$$

This encoding is efficient in terms of the number of qubits, since $n$ qubits can represent up to $2^n$ amplitudes. However, preparing an arbitrary amplitude-encoded state can be costly on real quantum hardware.

* **ZZ Feature Map**: this feature map encodes classical data using single-qubit rotations and two-qubit interaction terms based on Pauli-$Z$ operators. The main idea is to encode not only individual features, but also correlations between pairs of features through terms such as $x_i x_j$. This makes the map more expressive than simple angle encoding, although it usually requires deeper circuits with many two-qubit gates, which are more prone to errors on real quantum hardware.

Some examples of variational forms are:

* **Two-local**: this ansatz is built by alternating layers of parametrized single-qubit rotations and two-qubit entangling gates. For $n$ qubits and $k$ repetitions, it uses $n(k+1)$ trainable parameters, because it actually contains $k+1$ rotation layers. A typical version applies $R_y(\theta_{rj})$ gates to every qubit in each layer and CNOT gates between neighboring qubits to create entanglement. It is a versatile ansatz and can be combined with different measurement strategies.

* **Tree tensor**: this variational form follows a tree-like entanglement structure, where qubits are progressively combined through CNOT gates and parametrized rotations. For $n=2^k$ qubits, it contains $k+1$ layers, with fewer parameters at each new layer. This structure is usually shallower than fully connected ansatzes and is especially natural for binary classification tasks, where one can measure the expectation value of a single output qubit.

* **Strongly entangling layers**: this ansatz consists of repeated layers of general single-qubit rotations followed by entangling gates. Each layer usually applies rotations such as $R_z$, $R_y$ and $R_z$ to every qubit, giving $3nk$ trainable parameters for $n$ qubits and $k$ layers. The entangling gates connect qubits according to a chosen range or pattern, making the circuit highly expressive. However, this expressivity comes at the cost of deeper circuits and more two-qubit gates, which are more prone to noise on real quantum hardware.


Finally, as in classical neural networks, the optimization algorithm also plays an essential role in the model's training stage. We will use gradient descent algorithms with the Adam optimizer, so we need to compute the gradient of the expected value of a chosen loss function with respect to the optimizable parameters. The three main methods for calculating these gradients are:

* **Numerical approximation**: this method estimates the gradient by evaluating the model several times with slightly shifted parameter values. For a real-valued function $f:\mathbb{R}^n \rightarrow \mathbb{R}$, a simple finite-difference approximation of the partial derivative with respect to $x_j$ is

$$
\frac{\partial f}{\partial x_j}
\approx
\frac{f(x_1,\dots,x_j+h,\dots,x_n)-f(x_1,\dots,x_n)}{h},
$$

where $h$ is a small number. This method is general and easy to understand, but it can be inefficient because the quantum circuit must be evaluated many times.

* **Automatic differentiation**: when the QNN is executed on a simulator, classical automatic differentiation techniques can be used to compute gradients efficiently. This is similar to what is done in classical neural networks. The main advantage is that it can provide accurate gradients without manually deriving them. However, it is mainly useful in simulation, since real quantum hardware does not directly expose the internal state transformations needed for standard backpropagation.

* **Parameter shift rule**: this method allows gradients to be computed by running the same quantum circuit with shifted values of the trainable parameters. It is especially important because it can be used on real quantum hardware for many common parametrized gates. For example, if $E(\theta)$ is the expectation value obtained from a circuit containing a rotation gate depending on $\theta$, then its derivative can often be computed as

$$
\nabla_{\theta}E(\theta)
=
\frac{1}{2}
\left[
E\left(\theta+\frac{\pi}{2}\right)
-
E\left(\theta-\frac{\pi}{2}\right)
\right].
$$

This avoids purely numerical finite differences and provides an exact gradient formula for many standard quantum gates.



# Implementing QNNs with PennyLane

In [ ]:
dev = qml.